In [26]:
import re
import pandas as pd
from nba_api.stats import endpoints
from great_tables import gt

# Awards

## The Spark Plug Award (sponsored by Lt. Surge, presented by American Express CEO Stephen J Squeri)

Most charges drawn per 36 minutes, minimum 70% of games played (credit to morron88 for the idea to separate charges & loose balls in 2020)

In [27]:
compact_standings = (
    endpoints.leaguestandingsv3.LeagueStandingsV3()
    .get_data_frames()[0]
    [["TeamID","TeamCity","TeamName","Conference","Division","WINS","LOSSES"]]
    .assign(
        TeamGP=lambda d: d.WINS + d.LOSSES,
        TeamFullName=lambda d: d.TeamCity + " " + d.TeamName
    )
)

player_w_gp_percentages = (
    endpoints.leaguedashplayerbiostats.LeagueDashPlayerBioStats()
    .get_data_frames()[0]
    .loc[:, "PLAYER_ID":"GP"]
    .merge(
        compact_standings.filter(regex="Team"),
        left_on="TEAM_ID",
        right_on="TeamID",
        how="left"
    )
    .assign(G_PERCENT=lambda d: d.GP / d.TeamGP)
)

hustle = (
    endpoints.leaguehustlestatsplayer.LeagueHustleStatsPlayer()
    .get_data_frames()[0]
    .loc[:, lambda d: ~d.columns.str.contains("PCT")]
)

hustle_w_gp_qualify = (
    hustle
    .merge(
        player_w_gp_percentages[["PLAYER_ID","G_PERCENT"]],
        on="PLAYER_ID",
        how="left"
    )
)

per36_cols = hustle_w_gp_qualify.columns[
    hustle_w_gp_qualify.columns.get_loc("MIN")+1:
]

per36_cols = [c for c in per36_cols if c != "G_PERCENT"]

hustle_w_gp_qualify[[f"{c}_per_36" for c in per36_cols]] = (
    hustle_w_gp_qualify[per36_cols]
    .div(hustle_w_gp_qualify["MIN"], axis=0)
    .mul(36)
)

In [28]:
gt.GT(
        hustle_w_gp_qualify
        .loc[lambda d: d["G_PERCENT"] >= 0.7]
        .nlargest(5, "CHARGES_DRAWN_per_36", keep="all")
        [["PLAYER_NAME","TEAM_ABBREVIATION","MIN","CHARGES_DRAWN","CHARGES_DRAWN_per_36"]]
        .rename(columns={"PLAYER_NAME":"player","TEAM_ABBREVIATION":"team"})
    ).fmt_number(columns="CHARGES_DRAWN_per_36", decimals=4).cols_label_with(fn=lambda col: re.sub("_", " ", col))

player,team,MIN,CHARGES DRAWN,CHARGES DRAWN per 36
Jaylin Williams,OKC,1277.0,17,0.4792
Marcus Smart,LAL,1769.0,20,0.4070
Jalen Brunson,NYK,2590.0,29,0.4031
Dru Smith,MIA,1133.0,12,0.3813
Brandin Podziemski,GSW,2333.0,23,0.3549


## The Most Loose Balls Recovered Award (sponsored by Hungry Hungry Hippos, presented by Dennis Rodman & Nene's doctor)

Per 36 minutes, minimum 70% of games played

In [29]:
gt.GT(
        hustle_w_gp_qualify
        .loc[lambda d: d["G_PERCENT"] >= 0.7]
        .nlargest(5, "LOOSE_BALLS_RECOVERED_per_36", keep="all")
        [["PLAYER_NAME","TEAM_ABBREVIATION","MIN","LOOSE_BALLS_RECOVERED","LOOSE_BALLS_RECOVERED_per_36"]]
        .rename(columns={"PLAYER_NAME":"player","TEAM_ABBREVIATION":"team"})
    ).fmt_number(columns="LOOSE_BALLS_RECOVERED_per_36", decimals=4).cols_label_with(fn=lambda col: re.sub("_", " ", col))

player,team,MIN,LOOSE BALLS RECOVERED,LOOSE BALLS RECOVERED per 36
Josh Okogie,HOU,1334.0,47,1.2684
Cedric Coward,MEM,1598.0,54,1.2165
Javonte Green,DET,1428.0,48,1.2101
Ausar Thompson,DET,1896.0,62,1.1772
Paul Reed,DET,892.0,29,1.1704


## The Plexiglass Award

most deflections per 36 minutes, minimum 70% of games played

In [30]:
gt.GT(
    hustle_w_gp_qualify.loc[lambda d: d["G_PERCENT"] >= 0.7]
    .nlargest(5, "DEFLECTIONS_per_36", keep="all")[
        ["PLAYER_NAME", "TEAM_ABBREVIATION", "MIN", "DEFLECTIONS", "DEFLECTIONS_per_36"]
    ]
    .rename(columns={"PLAYER_NAME": "player", "TEAM_ABBREVIATION": "team"})
).fmt_number(columns="DEFLECTIONS_per_36", decimals=4).cols_label_with(
    fn=lambda col: re.sub("_", " ", col)
)

player,team,MIN,DEFLECTIONS,DEFLECTIONS per 36
Dru Smith,MIA,1133.0,196,6.2277
Ausar Thompson,DET,1896.0,317,6.0190
Jordan Goodwin,PHX,1572.0,260,5.9542
Cason Wallace,OKC,2046.0,338,5.9472
Javonte Green,DET,1428.0,207,5.2185


## The Wes Unseld Memorial Brick Wall Award

most points generated by screen assists per 36 minutes, minimum 70% of games played

In [31]:
gt.GT(
        hustle_w_gp_qualify
        .loc[lambda d: d["G_PERCENT"] >= 0.7]
        .nlargest(5, "SCREEN_AST_PTS_per_36", keep="all")
        [["PLAYER_NAME","TEAM_ABBREVIATION","MIN","SCREEN_AST_PTS","SCREEN_AST_PTS_per_36"]]
        .rename(columns={"PLAYER_NAME":"player","TEAM_ABBREVIATION":"team"})
    ).fmt_number(columns="SCREEN_AST_PTS_per_36", decimals=4).cols_label_with(fn=lambda col: re.sub("_", " ", col))

player,team,MIN,SCREEN AST PTS,SCREEN AST PTS per 36
Luke Kornet,SAS,1430.0,524,13.1916
Luka Garza,BOS,1116.0,388,12.5161
Neemias Queta,BOS,1926.0,666,12.4486
Oso Ighodaro,PHX,1808.0,613,12.2058
Marvin Bagley III,DAL,1201.0,407,12.1998


## The "He Trick Y'All, Running Around, Doing Nothing" Award (sponsored by Russell Westbrook, presented by Tony Snell)

Lowest mean of per-36 percentile ranks in the following: charges, contested shots, deflections, defensive boxouts, defensive loose balls recovered (minimum 50% of games played)

In [32]:
patterns = ["CHARGES", "2PT", "3PT", "DEFLECTIONS", "DEF_BOX", "DEF_LOOSE"]

hustle_50_percent_gp = (
    hustle_w_gp_qualify
    # filter rows
    .loc[lambda df: df['G_PERCENT'] >= 0.5]
    # select columns
    .pipe(lambda df: df[["PLAYER_NAME", "MIN"] + df.filter(regex="|".join(patterns)).columns.tolist()])
    # relocate per_36 columns to end
    .pipe(lambda df: df[[c for c in df.columns if 'per_36' not in c] + df.filter(like='per_36').columns.tolist()])
    # add percentile rank columns for per_36
    .pipe(lambda df: df.assign(**{f"{col}_percentile_rk": df[col].rank(pct=True) 
                                  for col in df.filter(like='per_36').columns}))
    # add sum of percentile rank columns
    .assign(mean=lambda df: df.filter(like='_percentile_rk').mean(axis=1))
)

hustle_50_percent_gp.to_csv('Output Data/Hustle Ranks.csv',index=False)


In [33]:
percentile_cols = [c for c in hustle_50_percent_gp.columns if "percentile_rk" in c]+["mean"]

gt.GT(
        hustle_50_percent_gp
        .nsmallest(5, "mean", keep="all")
        .loc[:, 
    ["PLAYER_NAME", "MIN"] + 
    [c for c in hustle_50_percent_gp.columns if "percentile_rk" in c] + ["mean"]
].rename(columns={"PLAYER_NAME": "player"})
).fmt_percent(
    columns=percentile_cols, decimals=2
    ).tab_spanner(
        label="Per 36 Percentile Rank", columns=percentile_cols
        ).cols_label_with(
        fn=lambda col: re.sub("_", " ", re.sub("per_36_percentile_rk", "", col))
    )

GT(_tbl_data=             player     MIN  ...  DEF_BOXOUTS_per_36_percentile_rk      mean
73       Caleb Love   968.0  ...                          0.060942  0.153740
342   Klay Thompson  1415.0  ...                          0.052632  0.164358
184  Gary Trent Jr.  1298.0  ...                          0.080332  0.175900
532     Tre Johnson  1401.0  ...                          0.171745  0.188366
556      Tyus Jones   832.0  ...                          0.174515  0.189289

[5 rows x 9 columns], _body=<great_tables._gt_data.Body object at 0x162725950>, _boxhead=Boxhead([ColInfo(var='player', type=<ColInfoTypeEnum.default: 1>, column_label='player', column_align='left', column_width=None), ColInfo(var='MIN', type=<ColInfoTypeEnum.default: 1>, column_label='MIN', column_align='right', column_width=None), ColInfo(var='CONTESTED_SHOTS_2PT_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CONTESTED SHOTS 2PT ', column_align='right', column_width=None), ColInfo(var='CONTESTED_SHOTS_3PT_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CONTESTED SHOTS 3PT ', column_align='right', column_width=None), ColInfo(var='DEFLECTIONS_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEFLECTIONS ', column_align='right', column_width=None), ColInfo(var='CHARGES_DRAWN_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CHARGES DRAWN ', column_align='right', column_width=None), ColInfo(var='DEF_LOOSE_BALLS_RECOVERED_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEF LOOSE BALLS RECOVERED ', column_align='right', column_width=None), ColInfo(var='DEF_BOXOUTS_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEF BOXOUTS ', column_align='right', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1564c6610>, _spanners=Spanners([SpannerInfo(spanner_id='Per 36 Percentile Rank', spanner_level=0, spanner_label='Per 36 Percentile Rank', spanner_units=None, spanner_pattern=None, vars=['CONTESTED_SHOTS_2PT_per_36_percentile_rk', 'CONTESTED_SHOTS_3PT_per_36_percentile_rk', 'DEFLECTIONS_per_36_percentile_rk', 'CHARGES_DRAWN_per_36_percentile_rk', 'DEF_LOOSE_BALLS_RECOVERED_per_36_percentile_rk', 'DEF_BOXOUTS_per_36_percentile_rk', 'mean'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1564a8510>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1564a8390>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1564a82d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1564ab650>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=Option

## The "Got that Dawg in Him" Award (presented by Air Bud)*

Highest sum of per-36 percentile ranks in the following: charges, contested shots, deflections, defensive boxouts, defensive loose balls recovered (minimum 50% of games played) (credit to memeticengineering for the idea)

In [34]:
gt.GT(
        hustle_50_percent_gp
        .nlargest(5, "mean", keep="all")
        .loc[:, 
    ["PLAYER_NAME", "MIN"] + [c for c in hustle_50_percent_gp.columns if "percentile_rk" in c] + ["mean"]
].rename(columns={"PLAYER_NAME": "player"})
).fmt_percent(
    columns=percentile_cols, decimals=2
    ).tab_spanner(
        label="Per 36 Percentile Rank", columns=percentile_cols
        ).cols_label_with(
        fn=lambda col: re.sub("_", " ", re.sub("per_36_percentile_rk", "", col))
    )

GT(_tbl_data=             player     MIN  ...  DEF_BOXOUTS_per_36_percentile_rk      mean
460       Paul Reed   892.0  ...                          0.872576  0.857572
411  Mouhamed Gueye  1160.0  ...                          0.914127  0.835642
165   Dwight Powell   899.0  ...                          0.941828  0.817636
355  Kyshawn George  1391.0  ...                          0.714681  0.798246
229    Jakob Poeltl  1149.0  ...                          0.950139  0.791782

[5 rows x 9 columns], _body=<great_tables._gt_data.Body object at 0x1564a8850>, _boxhead=Boxhead([ColInfo(var='player', type=<ColInfoTypeEnum.default: 1>, column_label='player', column_align='left', column_width=None), ColInfo(var='MIN', type=<ColInfoTypeEnum.default: 1>, column_label='MIN', column_align='right', column_width=None), ColInfo(var='CONTESTED_SHOTS_2PT_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CONTESTED SHOTS 2PT ', column_align='right', column_width=None), ColInfo(var='CONTESTED_SHOTS_3PT_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CONTESTED SHOTS 3PT ', column_align='right', column_width=None), ColInfo(var='DEFLECTIONS_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEFLECTIONS ', column_align='right', column_width=None), ColInfo(var='CHARGES_DRAWN_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='CHARGES DRAWN ', column_align='right', column_width=None), ColInfo(var='DEF_LOOSE_BALLS_RECOVERED_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEF LOOSE BALLS RECOVERED ', column_align='right', column_width=None), ColInfo(var='DEF_BOXOUTS_per_36_percentile_rk', type=<ColInfoTypeEnum.default: 1>, column_label='DEF BOXOUTS ', column_align='right', column_width=None), ColInfo(var='mean', type=<ColInfoTypeEnum.default: 1>, column_label='mean', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1564c6190>, _spanners=Spanners([SpannerInfo(spanner_id='Per 36 Percentile Rank', spanner_level=0, spanner_label='Per 36 Percentile Rank', spanner_units=None, spanner_pattern=None, vars=['CONTESTED_SHOTS_2PT_per_36_percentile_rk', 'CONTESTED_SHOTS_3PT_per_36_percentile_rk', 'DEFLECTIONS_per_36_percentile_rk', 'CHARGES_DRAWN_per_36_percentile_rk', 'DEF_LOOSE_BALLS_RECOVERED_per_36_percentile_rk', 'DEF_BOXOUTS_per_36_percentile_rk', 'mean'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1564dd2d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1564dda10>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1564dc410>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1564df090>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=Option

## The BBQ Chicken Award (presented by Shaquille O'Neal)

worst isolation defenders by points saved compared to league average, min 10 MPG & 10 possessions (credit to TitanTigers & RTLT512)

In [35]:
def add_league_avg_ppp(play_type):
    # Team-level data (for league average)
    team_df = endpoints.synergyplaytypes.SynergyPlayTypes(
        play_type_nullable=play_type,
        per_mode_simple="Totals",
        type_grouping_nullable="Defensive",
        player_or_team_abbreviation="T"
    ).get_data_frames()[0]

    league_avg_ppp = (
        team_df[['PTS', 'POSS']]
        .sum()
        .pipe(lambda x: x['PTS'] / x['POSS'] if x['POSS'] != 0 else 0)
    )

    # Player-level data
    player_df = endpoints.synergyplaytypes.SynergyPlayTypes(
        play_type_nullable=play_type,
        per_mode_simple="Totals",
        type_grouping_nullable="Defensive",
        player_or_team_abbreviation="P"
    ).get_data_frames()[0]

    # Add calculations
    player_df = player_df.assign(
        league_ppp=league_avg_ppp,
        player_ppp=lambda x: x['PTS'] / x['POSS'],
        pts_saved_above_lg_avg=lambda x: (x['league_ppp'] - x['player_ppp']) * x['POSS']
    )

    return player_df

In [36]:
iso_defense=add_league_avg_ppp("Isolation")

gt.GT(
    iso_defense.nsmallest(n=5,columns="pts_saved_above_lg_avg")
    [['PLAYER_NAME','TEAM_ABBREVIATION','POSS','player_ppp','pts_saved_above_lg_avg']].
    rename(columns={'PLAYER_NAME':'player','TEAM_ABBREVIATION':'team'})
).fmt_number(columns=['player_ppp','pts_saved_above_lg_avg'],decimals=3)

player,team,POSS,player_ppp,pts_saved_above_lg_avg
Thomas Bryant,CLE,28,2.071,−31.968
Kelly Oubre Jr.,PHI,43,1.395,−20.023
Spencer Jones,DEN,67,1.224,−19.710
Patrick Williams,CHI,45,1.356,−19.163
Pascal Siakam,IND,78,1.154,−17.483


## The Rotisserie Chicken Award (sponsored by Costco)

worst PNR ball handler defenders by points saved compared to league average, min 10 MPG & 10 possessions (credit to RTLT512 & altu_0002)

In [37]:
pnr_defense=add_league_avg_ppp("PRBallHandler")

gt.GT(
    pnr_defense.nsmallest(n=5,columns="pts_saved_above_lg_avg")
    [['PLAYER_NAME','TEAM_ABBREVIATION','POSS','player_ppp','pts_saved_above_lg_avg']].
    rename(columns={'PLAYER_NAME':'player','TEAM_ABBREVIATION':'team'})
).fmt_number(columns=['player_ppp','pts_saved_above_lg_avg'],decimals=3)

player,team,POSS,player_ppp,pts_saved_above_lg_avg
Ben Sheppard,IND,165,1.255,−62.049
Miles Bridges,CHA,270,1.052,−46.807
Maxime Raynaud,SAC,413,0.983,−43.182
Jaden McDaniels,MIN,457,0.969,−41.528
Jusuf Nurkić,UTA,207,1.077,−41.152


## The "Would You Mind Coming with Us, Sir?" Award (sponsored by the TSA, presented by Paul Blart)

worst screen defenders by points saved compared to league average, min 10 MPG & 10 possessions (credit to blackjack_trial)

In [38]:
screen_defense=add_league_avg_ppp("OffScreen")

gt.GT(
    screen_defense.nsmallest(n=5,columns="pts_saved_above_lg_avg")
    [['PLAYER_NAME','TEAM_ABBREVIATION','POSS','player_ppp','pts_saved_above_lg_avg']].
    rename(columns={'PLAYER_NAME':'player','TEAM_ABBREVIATION':'team'})
).fmt_number(columns=['player_ppp','pts_saved_above_lg_avg'],decimals=3)

player,team,POSS,player_ppp,pts_saved_above_lg_avg
Miles Bridges,CHA,52,1.250,−14.139
Moses Moody,GSW,36,1.361,−13.789
Isaiah Collier,UTA,30,1.400,−12.657
Nolan Traore,BKN,18,1.667,−12.394
Gary Harris,MIL,15,1.800,−12.329


In [39]:
playtype_defense_summary=pd.concat(
    [
        screen_defense[['PLAY_TYPE',"PLAYER_NAME","TEAM_ABBREVIATION","POSS","POSS_PCT",
        "player_ppp",'league_ppp','pts_saved_above_lg_avg']],
        iso_defense[['PLAY_TYPE',"PLAYER_NAME","TEAM_ABBREVIATION","POSS","POSS_PCT",
        "player_ppp",'league_ppp','pts_saved_above_lg_avg']],
        pnr_defense[['PLAY_TYPE',"PLAYER_NAME","TEAM_ABBREVIATION","POSS","POSS_PCT",
        "player_ppp",'league_ppp','pts_saved_above_lg_avg']]
    ]
)

playtype_defense_summary.to_csv('Output Data/Playtype Defense Summary.csv',index=False)

## The Trickshot Grenadier Award (presented by Dude Perfect)

Highest average of percentile ranks in FGA, FGA frequency & eFG% on shots with 4 seconds or less on the shotclock (credit to BehavioralSink & Bylanta for the idea)

In [40]:
trickshot_grenades=(
    endpoints.leaguedashplayerptshot.LeagueDashPlayerPtShot(
        shot_clock_range_nullable='4-0 Very Late'
    )
    .get_data_frames()[0]
    .loc[:, 'PLAYER_ID':'EFG_PCT']
    .assign(**{
        f"{c}_percentile": lambda d, c=c: d[c].rank(pct=True)
        for c in ["FGA_FREQUENCY", "FGA", "EFG_PCT"]
    })
    .assign(avg_percentiles=lambda d: d.filter(like="percentile").mean(axis=1))
)

trickshot_grenades.to_csv('Output Data/Trickshot Grenadier.csv',index=False)

In [41]:
gt.GT(
    trickshot_grenades.nlargest(5, "avg_percentiles", keep="all").loc[
        :, ["PLAYER_NAME","FGA_FREQUENCY","FGA","EFG_PCT"]
          + [c for c in trickshot_grenades.columns if c.endswith("percentile")] +
          ["avg_percentiles"]
    ]
).fmt_percent(
    columns=[c for c in trickshot_grenades.columns 
             if c not in ["PLAYER_NAME","FGA"] 
             and (c.endswith("percentile") or c in ["FGA_FREQUENCY","FGA","EFG_PCT","avg_percentiles"])]
).cols_label(
    FGA_FREQUENCY_percentile="FGA_FREQ",
    FGA_percentile="FGA",
    EFG_PCT_percentile="EFG_PCT"
).tab_spanner(
        label="Percentile Rank", 
        columns=[c for c in trickshot_grenades.columns if c.endswith("percentile")]
        )

GT(_tbl_data=           PLAYER_NAME  FGA_FREQUENCY  ...  EFG_PCT_percentile  avg_percentiles
39    Kevin Porter Jr.          0.169  ...            0.837887         0.908622
0     Payton Pritchard          0.188  ...            0.685792         0.886764
14  Brandin Podziemski          0.131  ...            0.782332         0.874317
4        Jalen Brunson          0.111  ...            0.823315         0.867638
22           Josh Hart          0.171  ...            0.642987         0.855191

[5 rows x 8 columns], _body=<great_tables._gt_data.Body object at 0x15473b210>, _boxhead=Boxhead([ColInfo(var='PLAYER_NAME', type=<ColInfoTypeEnum.default: 1>, column_label='PLAYER_NAME', column_align='left', column_width=None), ColInfo(var='FGA_FREQUENCY', type=<ColInfoTypeEnum.default: 1>, column_label='FGA_FREQUENCY', column_align='right', column_width=None), ColInfo(var='FGA', type=<ColInfoTypeEnum.default: 1>, column_label='FGA', column_align='right', column_width=None), ColInfo(var='EFG_PCT', type=<ColInfoTypeEnum.default: 1>, column_label='EFG_PCT', column_align='right', column_width=None), ColInfo(var='FGA_FREQUENCY_percentile', type=<ColInfoTypeEnum.default: 1>, column_label='FGA_FREQ', column_align='right', column_width=None), ColInfo(var='FGA_percentile', type=<ColInfoTypeEnum.default: 1>, column_label='FGA', column_align='right', column_width=None), ColInfo(var='EFG_PCT_percentile', type=<ColInfoTypeEnum.default: 1>, column_label='EFG_PCT', column_align='right', column_width=None), ColInfo(var='avg_percentiles', type=<ColInfoTypeEnum.default: 1>, column_label='avg_percentiles', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x160314850>, _spanners=Spanners([SpannerInfo(spanner_id='Percentile Rank', spanner_level=0, spanner_label='Percentile Rank', spanner_units=None, spanner_pattern=None, vars=['FGA_FREQUENCY_percentile', 'FGA_percentile', 'EFG_PCT_percentile'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1602fcb50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1602fca90>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1602fc9d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1602fc350>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo

## The "Imma Make 'Em Both" Award (presented by Diar DeRozan & Grant Williams)*

biggest decline from non-clutch FT% to clutch FT%, min 15 clutch FTA (credit to Necessary_Career_253 for the idea and midnightgreen29 for the name)

In [42]:
clutch_and_totals=(
    # totals
    endpoints.leaguedashplayerstats.LeagueDashPlayerStats().get_data_frames()[0]
    .loc[:,'PLAYER_ID':'PLUS_MINUS'].drop(columns='NICKNAME')
).merge(
    (  
        endpoints.leaguedashplayerclutch.LeagueDashPlayerClutch()
        .get_data_frames()[0]
        .loc[:, 'PLAYER_ID':'PLUS_MINUS']
        .drop(columns='NICKNAME')
        .rename(columns=lambda c: c if c in ['PLAYER_ID','PLAYER_NAME','TEAM_ID','TEAM_ABBREVIATION','AGE'] else c + '_clutch')
        ),how='left'
)

ft_div_by_clutch=(
    clutch_and_totals[['PLAYER_NAME','FTM','FTA','FTM_clutch','FTA_clutch']]
    .assign(
        FTM_non_clutch=lambda x: x.FTM-x.FTM_clutch,
        FTA_non_clutch=lambda x: x.FTA-x.FTA_clutch,
        FT_PCT_clutch=lambda x: x.FTM_clutch/x.FTA_clutch,
        FT_PCT_non_clutch=lambda x: x.FTM_non_clutch/x.FTA_non_clutch,
        differential=lambda x: x.FT_PCT_clutch-x.FT_PCT_non_clutch
    )
)

ft_div_by_clutch.to_csv('Output Data/Clutch Free Throw Differential.csv',index=False)

In [43]:
gt.GT(
    ft_div_by_clutch
    .query("FTA_clutch >= 15")
    .nsmallest(5, "differential",keep="all")
    [[
        "PLAYER_NAME",
        "FTA_clutch",
        "FT_PCT_clutch",
        "FT_PCT_non_clutch",
        "differential"
    ]]
).fmt_percent(
    columns=[
        "FT_PCT_clutch",
        "FT_PCT_non_clutch",
        "differential"
    ]
)

PLAYER_NAME,FTA_clutch,FT_PCT_clutch,FT_PCT_non_clutch,differential
Reed Sheppard,15.0,60.00%,84.51%,−24.51%
Donovan Clingan,17.0,47.06%,69.89%,−22.83%
Victor Wembanyama,38.0,63.16%,84.47%,−21.31%
Karl-Anthony Towns,16.0,68.75%,86.50%,−17.75%
Pascal Siakam,22.0,54.55%,70.22%,−15.68%


## The "Ice, Ice, Baby" Award (sponsored by Hisense, presented by Vanilla Ice)*

biggest improvement from non-clutch FT% to clutch FT%, min 15 clutch FTA

In [44]:
gt.GT(
    ft_div_by_clutch
    .query("FTA_clutch >= 15")
    .nlargest(5, "differential",keep="all")
    [[
        "PLAYER_NAME",
        "FTA_clutch",
        "FT_PCT_clutch",
        "FT_PCT_non_clutch",
        "differential"
    ]]
).fmt_percent(
    columns=[
        "FT_PCT_clutch",
        "FT_PCT_non_clutch",
        "differential"
    ]
)

PLAYER_NAME,FTA_clutch,FT_PCT_clutch,FT_PCT_non_clutch,differential
De'Aaron Fox,21.0,95.24%,74.22%,21.02%
Luka Dončić,21.0,90.48%,77.56%,12.91%
Julius Randle,22.0,90.91%,79.75%,11.16%
Miles Bridges,23.0,91.30%,81.30%,10.00%
Jarrett Allen,19.0,78.95%,70.25%,8.70%


## The "Paint Allergy" Award

players w/highest % of shots from outside paint, min 150 FGA & 50% of games played (credit to frosiano for the original idea of highest percentage of 3FGA of total FGA, and to Drummallumin for the revised idea of all shots outside of 15 feet)

In [45]:
non_paint_fga=(
    endpoints.leaguedashplayershotlocations.LeagueDashPlayerShotLocations()
    .get_data_frames()[0]
    # filter out columns where the first level is "Corner 3"
    .pipe(lambda d: d.loc[:, [col for col in d.columns if col[0] != "Corner 3"]])
    .pipe(lambda d: d.set_axis(
        [
            f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col
            for col in d.columns
        ],
        axis=1
    ))
    # compute totals and derived columns
    .pipe(lambda d: d.assign(
        tot_fga=lambda d: d.filter(like="_FGA").sum(axis=1),
        tot_fgm=lambda d: d.filter(like="_FGM").sum(axis=1)
    ))
    .pipe(lambda d: d.assign(
        non_paint_fga=d["tot_fga"] - d["Restricted Area_FGA"] - d["In The Paint (Non-RA)_FGA"],
        non_paint_fga_percentage=lambda d: d["non_paint_fga"] / d["tot_fga"]
    ))
    .rename(columns={
        '_PLAYER_ID': 'PLAYER_ID',
    })
)

In [46]:
gt.GT(
    non_paint_fga
    .merge(player_w_gp_percentages, how='left')
    .loc[lambda df: (df['tot_fga'] >= 150) & (df['G_PERCENT'] >= 0.5)]
    .nlargest(5, 'non_paint_fga_percentage')
    .loc[:, [
        'PLAYER_NAME',
        'TEAM_ABBREVIATION',
        'tot_fga',
        'non_paint_fga',
        'non_paint_fga_percentage'
    ]]
    .rename(columns={
        'PLAYER_NAME': 'player',
        'TEAM_ABBREVIATION': 'team'
    })
).fmt_percent(
    columns=["non_paint_fga_percentage"]
    )

player,team,tot_fga,non_paint_fga,non_paint_fga_percentage
Nicolas Batum,LAC,238.0,235.0,98.74%
AJ Green,MIL,618.0,580.0,93.85%
Sam Hauser,BOS,601.0,547.0,91.01%
Klay Thompson,DAL,728.0,639.0,87.77%
Gabe Vincent,ATL,216.0,186.0,86.11%


## The "Lumberjack" Award (sponsored by Paul Bunyan)

players w/highest % of shots from inside paint, max height of 6'3" & min 150 FGA & minimum 50% of games played (credit to Drummallumin for the idea)

In [47]:
gt.GT(
    non_paint_fga.merge(player_w_gp_percentages, how='left', on="PLAYER_ID")
    # filter
    .loc[lambda d: (
        (d['tot_fga'] >= 150) &
        (d['G_PERCENT'] >= 0.5) &
        (d['PLAYER_HEIGHT_INCHES'] <= 6*12 + 3)
    )]
    # mutate
    .assign(
        paint_fga=lambda d: d['tot_fga'] - d['non_paint_fga'],
        paint_fga_percentage=lambda d: 1 - d['non_paint_fga_percentage']
    )
    # slice_max
    .nlargest(5, 'paint_fga_percentage')
    # select + rename
    .loc[:, [
        'PLAYER_NAME',
        'TEAM_ABBREVIATION',
        'PLAYER_HEIGHT',
        'tot_fga',
        'paint_fga',
        'paint_fga_percentage'
    ]]
    .rename(columns={
        'PLAYER_NAME': 'player',
        'TEAM_ABBREVIATION': 'team'
    })
).fmt_percent(
    columns=["paint_fga_percentage"]
    )

player,team,PLAYER_HEIGHT,tot_fga,paint_fga,paint_fga_percentage
Tre Jones,CHI,6-1,617.0,464.0,75.20%
Gary Payton II,GSW,6-2,415.0,284.0,68.43%
Brandon Williams,DAL,6-1,617.0,397.0,64.34%
Jeremiah Fears,NOP,6-3,1006.0,601.0,59.74%
Craig Porter Jr.,CLE,6-1,258.0,148.0,57.36%


## The "FUCK OUTTA HERE, I GOT THAT SHIT" Award (presented by Carmelo Anthony)

Lowest contested rebound percentage, minimum 50% of games played

In [48]:
reb_w_gp_qualify = (
    endpoints.playerdashptreb.PlayerDashPtReb(
        team_id=0,
        player_id=0
    )
    .get_data_frames()[0]  # OverallRebounding
    # type_convert() equivalent (best effort in pandas)
    .pipe(lambda d: d.convert_dtypes())
    # left_join with selected columns
    .merge(
        player_w_gp_percentages[[
            "PLAYER_NAME",
            "TEAM_ABBREVIATION",
            "PLAYER_ID",
            "G_PERCENT",
            "PLAYER_HEIGHT",
            "PLAYER_HEIGHT_INCHES"
        ]],
        how="left",
        on="PLAYER_ID"
    )
)

In [49]:
gt.GT(
    reb_w_gp_qualify
    # filter
    .loc[lambda d: d["G_PERCENT"] >= 0.5]
    # slice_min
    .nsmallest(5, "C_REB_PCT")
    # select + rename
    .loc[:, [
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "PLAYER_HEIGHT",
        "C_REB_PCT"
    ]]
    .rename(columns={
        "PLAYER_NAME": "player",
        "TEAM_ABBREVIATION": "team"
    })
).fmt_percent(
    columns=["C_REB_PCT"]
    )

player,team,PLAYER_HEIGHT,C_REB_PCT
Tyus Jones,DEN,6-0,9.00%
Jordan McLaughlin,SAS,5-11,11.50%
Immanuel Quickley,TOR,6-2,12.00%
Luke Kennard,LAL,6-5,12.80%
Jevon Carter,ORL,6-0,12.80%


alternatively: restricting to players > 6 foot 6 inches in height

In [50]:
gt.GT(
    reb_w_gp_qualify
    # filter
    .loc[lambda d: (
        (d['G_PERCENT'] >= 0.5) &
        (d['PLAYER_HEIGHT_INCHES'] > 6*12 + 6)
    )]
    # slice_min
    .nsmallest(5, "C_REB_PCT")
    # select + rename
    .loc[:, [
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "PLAYER_HEIGHT",
        "C_REB_PCT"
    ]]
    .rename(columns={
        "PLAYER_NAME": "player",
        "TEAM_ABBREVIATION": "team"
    })
).fmt_percent(
    columns=["C_REB_PCT"]
    )

player,team,PLAYER_HEIGHT,C_REB_PCT
Sam Hauser,BOS,6-7,19.50%
Jarace Walker,IND,6-7,19.60%
Dillon Brooks,PHX,6-7,21.00%
Egor Dëmin,BKN,6-8,21.20%
Anthony Black,ORL,6-7,21.60%


## The "Glass Cleaner" Award (presented by Dennis Rodman, sponsored by Windex)

Highest contested rebound percentage, minimum 50% of games played

In [51]:
gt.GT(
    reb_w_gp_qualify
    # filter
    .loc[lambda d: d["G_PERCENT"] >= 0.5]
    # slice_min
    .nlargest(5, "C_REB_PCT")
    # select + rename
    .loc[:, [
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "PLAYER_HEIGHT",
        "C_REB_PCT"
    ]]
    .rename(columns={
        "PLAYER_NAME": "player",
        "TEAM_ABBREVIATION": "team"
    })
).fmt_percent(
    columns=["C_REB_PCT"]
    )

player,team,PLAYER_HEIGHT,C_REB_PCT
Luka Garza,BOS,6-10,61.60%
Mitchell Robinson,NYK,7-0,60.70%
Ryan Kalkbrenner,CHA,7-1,59.80%
Clint Capela,HOU,6-10,57.40%
Jakob Poeltl,TOR,7-0,56.50%


alternatively: restricting to players < 6 foot 7 inches in height

In [52]:
gt.GT(
    reb_w_gp_qualify
    # filter
    .loc[lambda d: (
        (d['G_PERCENT'] >= 0.5) &
        (d['PLAYER_HEIGHT_INCHES'] <= 6*12 + 6)
    )]
    # slice_min
    .nlargest(5, "C_REB_PCT")
    # select + rename
    .loc[:, [
        "PLAYER_NAME",
        "TEAM_ABBREVIATION",
        "PLAYER_HEIGHT",
        "C_REB_PCT"
    ]]
    .rename(columns={
        "PLAYER_NAME": "player",
        "TEAM_ABBREVIATION": "team"
    })
).fmt_percent(
    columns=["C_REB_PCT"]
    )

player,team,PLAYER_HEIGHT,C_REB_PCT
Zion Williamson,NOP,6-6,49.60%
Jaylen Clark,MIN,6-5,48.20%
Andrew Wiggins,MIA,6-6,43.20%
Julian Phillips,MIN,6-6,41.30%
Keldon Johnson,SAS,6-6,40.50%


## The "Fine, I'll Do It Myself" Award (sponsored by Thanos, presented by Allen Iverson)

Highest percentage of unassisted field goals, minimum 50% of games played

In [53]:
scoring_w_gp_qualify = (
    endpoints.leaguedashplayerstats.LeagueDashPlayerStats(measure_type_detailed_defense='Scoring')
    .get_data_frames()[0]
    .pipe(lambda df: df.merge(player_w_gp_percentages, how='left'))
    .query("G_PERCENT >= 0.5")
)

In [54]:
gt.GT(
    scoring_w_gp_qualify
    .nlargest(5, 'PCT_UAST_FGM')  # equivalent to slice_max
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP', 'MIN', 'FGM', 'PCT_UAST_FGM']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(
    columns=["PCT_UAST_FGM"]
    )

player,team,GP,MIN,FGM,PCT_UAST_FGM
Shai Gilgeous-Alexander,OKC,68,33.2,731,80.30%
James Harden,CLE,70,34.8,487,75.40%
Luka Dončić,LAL,64,35.8,693,72.20%
Jalen Brunson,NYK,74,35.0,689,68.90%
T.J. McConnell,IND,56,17.2,242,66.50%


## The "You Gotta Feed Me" Award (presented by Joey Chestnut & Marcin Gortat)

Highest percentage of assisted field goals, minimum 50% of games played and 1 FGM per game

In [55]:
gt.GT(
    scoring_w_gp_qualify
    .query('FGM/GP>=1')
    .nlargest(5, 'PCT_AST_FGM')  # equivalent to slice_max
    .loc[:, ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'GP', 'MIN', 'FGM', 'PCT_AST_FGM']]  # select columns
    .rename(columns={'PLAYER_NAME': 'player', 'TEAM_ABBREVIATION': 'team'})  # rename like select()
).fmt_percent(
    columns=["PCT_AST_FGM"]
    )

player,team,GP,MIN,FGM,PCT_AST_FGM
Nicolas Batum,LAC,74,17.5,96,99.00%
Jaylin Williams,OKC,65,19.7,149,96.60%
Sam Hauser,BOS,78,24.8,252,92.90%
Vít Krejčí,POR,65,21.4,192,92.20%
Jamison Battle,TOR,61,8.5,74,91.90%
